# Part D: Multi-Seed Statistical Validation of MITR
## Experiment #1 from Part C — Scoped-Down Implementation

**Hypothesis:** If we run each MITR configuration with multiple random seeds, then
the reported improvements (or degradations) will be confirmed as statistically
significant (p < 0.05), because the effect sizes are large enough to survive
run-to-run variance.

### Scoped-down design (Note: The changes for running on T4 were commented in, uncomment and replace to run on T4)
- **Model:** DistilBERT (66M params)
- **Seeds:** 5 (42, 123, 456, 789, 1024)
- **Epochs:** 5
- **Configs:** Baseline, MITR-InfoNCE, MITR-Cosine, MITR-CKA, MITR-CLUB (all 4 strategies)
- **Precision:** FP16 (T4 does not support BF16)
- **Data:** BoolQ, 8K train / 1.5K val (unchanged)
- **Metrics:** Mean ± std for accuracy and contradiction rate across seeds;
  paired t-tests (Baseline vs each MITR strategy)

### Full experiment (what this would look like at scale)
5 seeds × all 4 strategies × 3 backbones (DistilBERT, BERT, RoBERTa) × 5 epochs ≈ 15–20 GPU hours on T4.


In [ ]:
# Run once - installs / upgrades required packages
!pip install -q --upgrade "transformers>=4.36" datasets accelerate matplotlib seaborn tqdm scipy
print("Done.")


In [ ]:
import os, json, random, warnings
from dataclasses import dataclass
from typing import Dict, List, Optional
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    DistilBertModel,
    DistilBertTokenizerFast,
    get_cosine_schedule_with_warmup,
)
from datasets import load_dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")


In [ ]:
SEEDS = [42, 123, 456, 789, 1024]
MI_STRATEGIES = ["infonce", "cosine", "cka", "club"]

@dataclass
class Config:
    # ── Model ─────────────────────────────────────────────────
    model_name: str  = "distilbert-base-uncased"
    num_labels: int  = 2          # 0 = False/No,  1 = True/Yes

    # ── Training ──────────────────────────────────────────────
    batch_size: int       = 64
    eval_batch_size: int  = 128
    learning_rate: float  = 2e-5
    weight_decay: float   = 0.01
    num_epochs: int       = 5
    warmup_ratio: float   = 0.06
    max_grad_norm: float  = 1.0
    grad_accum_steps: int = 1

    # ── MITR ──────────────────────────────────────────────────
    mi_lambda: float     = 0.01
    mi_warmup_steps: int = 200
    club_hidden: int     = 384
    mi_strategy: str     = "infonce"

    # ── Data ──────────────────────────────────────────────────
    dataset: str         = "boolq"
    max_length: int      = 256
    num_train: int       = 8000
    num_eval: int        = 1500
    num_contra: int      = 500
    num_workers: int     = 2      # T4 has fewer CPU cores than A100

    # ── T4 ────────────────────────────────────────────────────
    use_bf16: bool       = False  # T4 does not support BF16; use FP16 instead
    use_fp16: bool       = True
    compile_model: bool  = False  # torch.compile unreliable on T4
    seed: int            = 42

    # ── Output ────────────────────────────────────────────────
    output_dir: str      = "experiment_results"

CFG = Config()
print(CFG)
print(f"\nSeeds: {SEEDS}")
print(f"Strategies: {MI_STRATEGIES}")
print(f"Configs per seed: Baseline + {len(MI_STRATEGIES)} MITR = {1 + len(MI_STRATEGIES)}")
print(f"Total runs: {len(SEEDS)} seeds × {1 + len(MI_STRATEGIES)} configs = {len(SEEDS) * (1 + len(MI_STRATEGIES))}")

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)

torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if CFG.use_fp16 and DEVICE.type == "cuda":
    DTYPE = torch.float16
elif CFG.use_bf16 and DEVICE.type == "cuda" and torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float32

print(f"Device : {DEVICE}")
print(f"Dtype  : {DTYPE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")
    print(f"SMs    : {props.multi_processor_count}")

os.makedirs(CFG.output_dir, exist_ok=True)


In [ ]:
# BoolQ: yes/no QA about Wikipedia passages
# Fields: question (str), passage (str), answer (bool)

def load_boolq():
    ds = load_dataset("google/boolq")
    def fmt(ex):
        return {
            "text":     ex["question"] + " [SEP] " + ex["passage"][:400],
            "label":    int(ex["answer"]),
            "question": ex["question"],
        }
    train = ds["train"].map(fmt, remove_columns=ds["train"].column_names)
    val   = ds["validation"].map(fmt, remove_columns=ds["validation"].column_names)
    return train, val


# RuleTaker: rule-based logical inference with yes/no questions
# Fields: context (str), question (str), answer (bool)

def load_ruletaker():
    try:
        ds = load_dataset("tasksource/ruletaker")
        def fmt(ex):
            return {
                "text":     ex["context"][:500] + " [SEP] " + ex["question"],
                "label":    int(ex["answer"]),
                "question": ex["question"],
            }
        train   = ds["train"].map(fmt, remove_columns=ds["train"].column_names)
        val_key = "validation" if "validation" in ds else "test"
        val     = ds[val_key].map(fmt, remove_columns=ds[val_key].column_names)
        return train, val
    except Exception as exc:
        print(f"[WARN] RuleTaker unavailable ({exc}). Falling back to BoolQ.")
        return load_boolq()


print(f"Loading: {CFG.dataset} ...")
if CFG.dataset == "ruletaker":
    train_raw, val_raw = load_ruletaker()
else:
    train_raw, val_raw = load_boolq()

rng = random.Random(CFG.seed)

def subsample(dataset, n):
    if n < 0 or n >= len(dataset):
        return dataset
    return dataset.select(rng.sample(range(len(dataset)), n))

train_raw = subsample(train_raw, CFG.num_train)
val_raw   = subsample(val_raw,   CFG.num_eval)

tl = [x["label"] for x in train_raw]
vl = [x["label"] for x in val_raw]
print(f"Train : {len(train_raw):,}  (True={sum(tl)}, False={len(tl)-sum(tl)})")
print(f"Val   : {len(val_raw):,}  (True={sum(vl)}, False={len(vl)-sum(vl)})")
print(f"Sample: {repr(train_raw[0]['text'][:150])}")


In [ ]:
# Logical negation of yes/no questions
# e.g.  "is water wet"       ->  "is water not wet"
# e.g.  "are cats mammals"   ->  "are cats not mammals"
#
# Contradiction rate = fraction of pairs where model gives the SAME
# prediction to both q and NOT-q (logically impossible: they have opposite truth values).

_AUX = [
    ("is ",     "is not "),
    ("are ",    "are not "),
    ("was ",    "was not "),
    ("were ",   "were not "),
    ("does ",   "does not "),
    ("do ",     "do not "),
    ("did ",    "did not "),
    ("has ",    "has not "),
    ("have ",   "have not "),
    ("had ",    "had not "),
    ("can ",    "cannot "),
    ("could ",  "could not "),
    ("will ",   "will not "),
    ("would ",  "would not "),
    ("should ", "should not "),
]

def negate_question(q: str) -> Optional[str]:
    # Returns negated question string, or None if pattern not found
    q = q.strip().rstrip("?").lower()
    for pos, neg in _AUX:
        if q.startswith(neg):          # already negated -> strip negation
            return pos + q[len(neg):]
        if q.startswith(pos):
            return neg + q[len(pos):]
    return None


def create_contradiction_pairs(dataset, n_pairs: int) -> List[Dict]:
    pairs: List[Dict] = []
    for ex in dataset:
        q = ex.get("question", "").strip()
        if not q:
            continue
        q_neg = negate_question(q)
        if q_neg is None or q_neg.strip() == q.strip():
            continue

        text_fwd = ex["text"]
        if "[SEP]" in text_fwd:
            prefix   = text_fwd.split("[SEP]")[0]
            text_neg = prefix + "[SEP] " + q_neg
        else:
            text_neg = q_neg

        pairs.append({
            "text_forward":    text_fwd,
            "text_negated":    text_neg,
            "label_forward":   ex["label"],
            "label_negated":   1 - ex["label"],
            "question":        q,
            "negated_question": q_neg,
        })
        if len(pairs) >= n_pairs:
            break
    return pairs


# Build pairs from val set only (no data leakage)
contradiction_pairs = create_contradiction_pairs(val_raw, n_pairs=CFG.num_contra)
print(f"Contradiction pairs: {len(contradiction_pairs):,}")

p = contradiction_pairs[0]
print(f"  Forward : {repr(p['question'])}  -> label={p['label_forward']}")
print(f"  Negated : {repr(p['negated_question'])}  -> label={p['label_negated']}")


In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(CFG.model_name)


class LogicDataset(Dataset):
    # Binary classification dataset (single text input)
    def __init__(self, data, tokenizer, max_length: int):
        enc = tokenizer(
            [ex["text"] for ex in data],
            max_length=max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels         = torch.tensor([ex["label"] for ex in data], dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }


class ContradictionPairDataset(Dataset):
    # Paired (forward, negated) questions for contradiction evaluation
    def __init__(self, pairs: List[Dict], tokenizer, max_length: int):
        fwd = tokenizer([p["text_forward"] for p in pairs],
                        max_length=max_length, padding="max_length",
                        truncation=True, return_tensors="pt")
        neg = tokenizer([p["text_negated"]  for p in pairs],
                        max_length=max_length, padding="max_length",
                        truncation=True, return_tensors="pt")
        self.fwd_ids  = fwd["input_ids"];     self.fwd_mask = fwd["attention_mask"]
        self.neg_ids  = neg["input_ids"];     self.neg_mask = neg["attention_mask"]
        self.fwd_lbl  = torch.tensor([p["label_forward"] for p in pairs], dtype=torch.long)
        self.neg_lbl  = torch.tensor([p["label_negated"] for p in pairs], dtype=torch.long)

    def __len__(self):
        return len(self.fwd_lbl)

    def __getitem__(self, idx):
        return {
            "fwd_input_ids":      self.fwd_ids[idx],
            "fwd_attention_mask": self.fwd_mask[idx],
            "neg_input_ids":      self.neg_ids[idx],
            "neg_attention_mask": self.neg_mask[idx],
            "fwd_label":          self.fwd_lbl[idx],
            "neg_label":          self.neg_lbl[idx],
        }


# Build datasets
train_dataset = LogicDataset(train_raw, tokenizer, CFG.max_length)
val_dataset   = LogicDataset(val_raw,   tokenizer, CFG.max_length)
pair_dataset  = ContradictionPairDataset(contradiction_pairs, tokenizer, CFG.max_length)

# DataLoader kwargs optimised for A100
_dl_kw = dict(
    pin_memory=(DEVICE.type == "cuda"),
    num_workers=CFG.num_workers,
    persistent_workers=(CFG.num_workers > 0),
)
train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size,     shuffle=True,  **_dl_kw)
val_loader   = DataLoader(val_dataset,   batch_size=CFG.eval_batch_size, shuffle=False, **_dl_kw)
pair_loader  = DataLoader(pair_dataset,  batch_size=64,                  shuffle=False, **_dl_kw)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Pair  batches : {len(pair_loader)}")


In [ ]:
# CLUB (Contrastive Log-ratio Upper Bound) estimates I(X; Y) as an upper bound.
# Minimising the output pushes I(X; Y) down.
#
# Improvements over the original (Cheng et al., 2020):
#   - LayerNorm on hidden activations -> stable gradients
#   - logvar clamped to [-5, 2]  ->  exp in [0.007, 7.4]  (prevents division-by-near-zero)
#   - Output clamped to [-50, 50] to block gradient explosions

class CLUBSample(nn.Module):
    def __init__(self, x_dim: int, y_dim: int, hidden: int, dropout: float = 0.1):
        super().__init__()
        def mlp(in_d, out_d):
            return nn.Sequential(
                nn.Linear(in_d, hidden), nn.LayerNorm(hidden),
                nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, out_d),
            )
        self.p_mu     = mlp(x_dim, y_dim)
        self.p_logvar = mlp(x_dim, y_dim)

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        # x: (B, x_dim)  y: (B, y_dim)
        # Returns scalar upper-bound estimate of I(X; Y). Minimise to reduce MI.
        mu     = self.p_mu(x)
        logvar = self.p_logvar(x).clamp(-5.0, 2.0)
        var    = logvar.exp().clamp(min=1e-6)

        perm = torch.randperm(x.size(0), device=x.device)
        pos  = -((mu - y)       ** 2) / var   # paired    (B, y_dim)
        neg  = -((mu - y[perm]) ** 2) / var   # shuffled  (B, y_dim)

        upper_bound = (pos.sum(-1) - neg.sum(-1)).mean() / 2.0
        return upper_bound.clamp(-50.0, 50.0)

print("CLUBSample defined.")


## Additional MI Estimators

We compare **four** mutual-information proxy strategies to penalise layer redundancy:

| Strategy | Key idea | Invariances |
|----------|----------|-------------|
| **CLUB** | Parametric Gaussian upper bound on I(X;Y) | None (learned) |
| **InfoNCE** | Contrastive lower bound via cross-entropy over similarity matrix | Learned projections |
| **Cosine** | Mean pairwise cosine similarity as MI proxy | Scale-invariant (L2 norm) |
| **CKA** | Centered Kernel Alignment - compares representational *geometry* across the batch | Orthogonal transforms + isotropic scaling |

CKA is the most principled for detecting structural redundancy: two layers that encode the same information in a rotated basis will have low cosine similarity but **high CKA**.

In [ ]:
# ── InfoNCE MI Estimator ───────────────────────────────────────
# Contrastive lower bound: projects X and Y into a shared space,
# builds a similarity matrix, and uses cross-entropy to estimate MI.
# I(X;Y) >= log(N) - L_NCE

class InfoNCEMI(nn.Module):
    def __init__(self, x_dim: int, y_dim: int, hidden: int, temperature: float = 0.07):
        super().__init__()
        self.proj_x = nn.Sequential(
            nn.Linear(x_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden),
        )
        self.proj_y = nn.Sequential(
            nn.Linear(y_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden),
        )
        self.temperature = temperature

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        x_proj = F.normalize(self.proj_x(x), dim=-1)
        y_proj = F.normalize(self.proj_y(y), dim=-1)

        sim_matrix = torch.matmul(x_proj, y_proj.T) / self.temperature  # (B, B)
        labels = torch.arange(x.size(0), device=x.device)
        loss = F.cross_entropy(sim_matrix, labels)

        mi_estimate = torch.log(torch.tensor(float(x.size(0)), device=x.device)) - loss
        return mi_estimate.clamp(-20.0, 20.0)


# ── Cosine Similarity MI Proxy ─────────────────────────────────
# Simplest and most stable: mean cosine similarity across the batch.
# Higher similarity => more redundancy => minimise this.

class CosineSimMI(nn.Module):
    def __init__(self, x_dim: int, y_dim: int, hidden: int):
        super().__init__()
        # No learnable parameters needed when dims match
        pass

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        x_norm = F.normalize(x, dim=-1)
        y_norm = F.normalize(y, dim=-1)
        cos_sim = (x_norm * y_norm).sum(dim=-1).mean()
        return cos_sim.clamp(-10.0, 10.0)


# ── CKA (Centered Kernel Alignment) MI Proxy ──────────────────
# Compares representational *geometry* across the batch using Gram matrices.
# Invariant to orthogonal transforms and isotropic scaling - catches
# redundancy that cosine similarity misses (e.g. rotated bases).
#
# Linear CKA:  HSIC(X,Y) / sqrt(HSIC(X,X) * HSIC(Y,Y))
# where HSIC is estimated via ||X^T Y||_F^2 after centering.

class CKAMI(nn.Module):
    def __init__(self, x_dim: int = 768, y_dim: int = 768, hidden: int = 384):
        super().__init__()
        # No learnable parameters - CKA is a closed-form statistic.

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        # x, y: (B, D) - batch of layer residual diffs
        # Center both representations (subtract batch mean)
        x = x - x.mean(dim=0, keepdim=True)
        y = y - y.mean(dim=0, keepdim=True)

        # Cross-covariance and self-covariance Frobenius norms
        cross = torch.norm(x.t() @ y) ** 2        # ||X^T Y||_F^2
        self_x = torch.norm(x.t() @ x)            # ||X^T X||_F
        self_y = torch.norm(y.t() @ y)            # ||Y^T Y||_F

        cka = cross / (self_x * self_y + 1e-8)
        return cka.clamp(-10.0, 10.0)


# ── Factory ────────────────────────────────────────────────────

MI_ESTIMATORS = {
    "club":    CLUBSample,
    "infonce": InfoNCEMI,
    "cosine":  CosineSimMI,
    "cka":     CKAMI,
}

print("All MI estimators defined:", list(MI_ESTIMATORS.keys()))

In [ ]:
# Standard DistilBERT fine-tuned for binary classification.
# No MI regularisation - used as the comparison baseline.

class BaselineClassifier(nn.Module):
    def __init__(self, model_name: str = "distilbert-base-uncased", num_labels: int = 2):
        super().__init__()
        self.distilbert     = DistilBertModel.from_pretrained(model_name)
        self.pre_classifier = nn.Linear(768, 768)
        self.classifier     = nn.Linear(768, num_labels)
        self.dropout        = nn.Dropout(0.1)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        labels:         Optional[torch.Tensor] = None,
        is_training:    bool = False,      # unused; kept for uniform interface with MITR
    ) -> Dict:
        out = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]                   # CLS token  (B, 768)
        cls = self.dropout(F.relu(self.pre_classifier(cls)))
        logits = self.classifier(cls)                        # (B, num_labels)

        result = {"logits": logits, "mi_loss": 0.0}
        if labels is not None:
            result["loss"] = F.cross_entropy(logits, labels)
        return result

print("BaselineClassifier defined.")


In [ ]:
# DistilBERT with MITR regularisation.
#
# Core idea:
#   Each transformer layer's residual diff  diff[i] = H[i+1] - H[i]
#   encodes what that layer contributes.  High MI between consecutive diffs
#   means adjacent layers are doing redundant work.
#
#   MITR penalises this:
#       loss = (1 - lam) * task_loss  +  lam * mean_i MI(diff[i]; diff[i+1])
#
# MI Strategy:
#   Selectable via cfg.mi_strategy: "club", "infonce", "cosine", or "cka".

class MITRClassifier(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

        # Backbone - identical to BaselineClassifier
        self.distilbert     = DistilBertModel.from_pretrained(cfg.model_name)
        self.pre_classifier = nn.Linear(768, 768)
        self.classifier     = nn.Linear(768, cfg.num_labels)
        self.dropout        = nn.Dropout(0.1)

        # MI estimator - selected by cfg.mi_strategy
        estimator_cls = MI_ESTIMATORS[cfg.mi_strategy]
        self.mi_estimator = estimator_cls(768, 768, cfg.club_hidden)
        self._step = 0   # training step counter for lambda warmup

    def _effective_lambda(self) -> float:
        if self._step >= self.cfg.mi_warmup_steps:
            return self.cfg.mi_lambda
        return self.cfg.mi_lambda * (self._step / max(1, self.cfg.mi_warmup_steps))

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        labels:         Optional[torch.Tensor] = None,
        is_training:    bool = False,
    ) -> Dict:
        out = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

        cls    = out.last_hidden_state[:, 0]
        cls    = self.dropout(F.relu(self.pre_classifier(cls)))
        logits = self.classifier(cls)

        result = {"logits": logits, "mi_loss": 0.0}

        if labels is not None:
            task_loss = F.cross_entropy(logits, labels)
            lam       = self._effective_lambda()

            if is_training and lam > 0.0:
                hs = out.hidden_states

                diffs = []
                for i in range(len(hs) - 1):
                    d = (hs[i + 1] - hs[i]).mean(dim=1)
                    d = F.layer_norm(d, (d.size(-1),))
                    diffs.append(d)

                mi_list = [
                    self.mi_estimator(diffs[i], diffs[i + 1])
                    for i in range(len(diffs) - 1)
                ]

                if mi_list:
                    mi_mean          = torch.stack(mi_list).mean()
                    result["mi_loss"] = mi_mean.item()
                    result["loss"]    = (1.0 - lam) * task_loss + lam * mi_mean
                else:
                    result["loss"] = task_loss
            else:
                result["loss"] = task_loss

            if is_training:
                self._step += 1

        return result

print("MITRClassifier defined (supports: club, infonce, cosine, cka).")

In [ ]:
# ── Optimizer & scheduler ──────────────────────────────────────

def build_optimizer_scheduler(model: nn.Module, train_loader, cfg: Config):
    no_decay = {"bias", "LayerNorm.weight"}
    grouped  = [
        {"params": [p for n, p in model.named_parameters()
                    if p.requires_grad and not any(nd in n for nd in no_decay)],
         "weight_decay": cfg.weight_decay},
        {"params": [p for n, p in model.named_parameters()
                    if p.requires_grad and any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    try:
        opt = torch.optim.AdamW(grouped, lr=cfg.learning_rate, fused=True)
    except TypeError:
        opt = torch.optim.AdamW(grouped, lr=cfg.learning_rate)

    total_steps  = len(train_loader) * cfg.num_epochs // cfg.grad_accum_steps
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    sched        = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    return opt, sched


# ── Training loop ──────────────────────────────────────────────

def train_one_epoch(
    model, loader, optimizer, scheduler,
    device, dtype,
    grad_accum: int   = 1,
    max_grad_norm: float = 1.0,
    is_mitr: bool     = False,
) -> Dict:
    model.train()
    total_loss = total_mi = n = 0
    use_amp    = (dtype != torch.float32)

    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc="  train", leave=False)):
        ids  = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        lbls = batch["labels"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
            out = model(ids, mask, labels=lbls, is_training=is_mitr)

        (out["loss"] / grad_accum).backward()

        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        total_loss += out["loss"].item()
        mi_val      = out.get("mi_loss", 0.0)
        total_mi   += mi_val if isinstance(mi_val, float) else float(mi_val)
        n          += 1

    return {"train_loss": total_loss / n, "train_mi_loss": total_mi / n}


# ── Experiment 1 - Accuracy ────────────────────────────────────

@torch.no_grad()
def eval_accuracy(model, loader, device, dtype) -> Dict:
    model.eval()
    correct = total = 0; val_loss = 0.0
    use_amp = (dtype != torch.float32)

    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
            out = model(ids, mask, labels=lbls)
        preds    = out["logits"].argmax(-1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
        if "loss" in out:
            val_loss += out["loss"].item()

    return {"accuracy": correct / total, "val_loss": val_loss / len(loader)}


# ── Experiment 2 - Contradiction rate ─────────────────────────

@torch.no_grad()
def eval_contradiction_rate(model, pair_loader, device, dtype) -> Dict:
    model.eval()
    contradictions = fwd_correct = neg_correct = total = 0
    use_amp = (dtype != torch.float32)

    for batch in pair_loader:
        fwd_ids  = batch["fwd_input_ids"].to(device)
        fwd_mask = batch["fwd_attention_mask"].to(device)
        neg_ids  = batch["neg_input_ids"].to(device)
        neg_mask = batch["neg_attention_mask"].to(device)
        fwd_lbl  = batch["fwd_label"].to(device)
        neg_lbl  = batch["neg_label"].to(device)

        with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
            fwd_out = model(fwd_ids, fwd_mask)
            neg_out = model(neg_ids, neg_mask)

        fwd_pred = fwd_out["logits"].argmax(-1)
        neg_pred = neg_out["logits"].argmax(-1)

        contradictions += (fwd_pred == neg_pred).sum().item()
        fwd_correct    += (fwd_pred == fwd_lbl).sum().item()
        neg_correct    += (neg_pred == neg_lbl).sum().item()
        total          += fwd_lbl.size(0)

    return {
        "contradiction_rate": contradictions / total,
        "consistency_rate":   1.0 - contradictions / total,
        "fwd_accuracy":       fwd_correct / total,
        "neg_accuracy":       neg_correct / total,
    }

print("Utilities defined.")

In [ ]:
def run_experiment(name: str, model: nn.Module, cfg: Config, is_mitr: bool) -> Dict:
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    model = model.to(DEVICE)
    opt, sched = build_optimizer_scheduler(model, train_loader, cfg)
    history    = defaultdict(list)

    for epoch in range(1, cfg.num_epochs + 1):
        tr  = train_one_epoch(model, train_loader, opt, sched,
                               DEVICE, DTYPE,
                               grad_accum=cfg.grad_accum_steps,
                               max_grad_norm=cfg.max_grad_norm,
                               is_mitr=is_mitr)
        val = eval_accuracy(model, val_loader, DEVICE, DTYPE)

        history["epoch"].append(epoch)
        history["train_loss"].append(tr["train_loss"])
        history["train_mi_loss"].append(tr["train_mi_loss"])
        history["val_loss"].append(val["val_loss"])
        history["val_accuracy"].append(val["accuracy"])

        print(f"  Ep {epoch}/{cfg.num_epochs}  "
              f"train={tr['train_loss']:.4f}  mi={tr['train_mi_loss']:.4f}  "
              f"val={val['val_loss']:.4f}  acc={val['accuracy']:.4f}")

    print("\n  Evaluating contradiction rate ...")
    contra = eval_contradiction_rate(model, pair_loader, DEVICE, DTYPE)
    print(f"  Contradiction rate : {contra['contradiction_rate']:.4f}")
    print(f"  Consistency  rate  : {contra['consistency_rate']:.4f}")

    del model
    torch.cuda.empty_cache()

    return {
        "name":               name,
        "history":            dict(history),
        "final_accuracy":     history["val_accuracy"][-1],
        "final_val_loss":     history["val_loss"][-1],
        "contradiction_rate": contra["contradiction_rate"],
        "consistency_rate":   contra["consistency_rate"],
        "fwd_accuracy":       contra["fwd_accuracy"],
        "neg_accuracy":       contra["neg_accuracy"],
    }


# ── Multi-seed experiment loop ────────────────────────────────
# 5 seeds × 5 configs (Baseline + 4 MI strategies)

CONFIG_NAMES = ["baseline"] + [f"mitr_{s}" for s in MI_STRATEGIES]
multi_seed_results = {name: [] for name in CONFIG_NAMES}

total_runs = len(SEEDS) * len(CONFIG_NAMES)
run_num = 0

for seed in SEEDS:
    print(f"\n{'#'*60}")
    print(f"  SEED = {seed}")
    print(f"{'#'*60}")

    # Baseline
    run_num += 1
    print(f"\n  [{run_num}/{total_runs}] Baseline, seed={seed}")
    set_seed(seed)
    baseline_model = BaselineClassifier(CFG.model_name, CFG.num_labels)
    res = run_experiment(f"Baseline (seed={seed})", baseline_model, CFG, is_mitr=False)
    res["seed"] = seed
    multi_seed_results["baseline"].append(res)

    # MITR strategies
    for strategy in MI_STRATEGIES:
        run_num += 1
        print(f"\n  [{run_num}/{total_runs}] MITR-{strategy.upper()}, seed={seed}")
        set_seed(seed)
        cfg_s = Config()
        cfg_s.mi_strategy = strategy
        cfg_s.seed = seed
        model = MITRClassifier(cfg_s)
        res = run_experiment(
            f"MITR-{strategy.upper()} (seed={seed})", model, cfg_s, is_mitr=True
        )
        res["seed"] = seed
        multi_seed_results[f"mitr_{strategy}"].append(res)

# ── Save raw results ──────────────────────────────────────────
out_json = os.path.join(CFG.output_dir, "multi_seed_results.json")
with open(out_json, "w") as fh:
    json.dump(multi_seed_results, fh, indent=2, default=float)
print(f"\nAll {total_runs} runs complete. Results saved -> {out_json}")

In [ ]:
# ── Aggregate multi-seed results ──────────────────────────────

def aggregate(results_list):
    accs   = [r["final_accuracy"] for r in results_list]
    contras = [r["contradiction_rate"] for r in results_list]
    return {
        "acc_mean": np.mean(accs),  "acc_std": np.std(accs, ddof=1),
        "acc_vals": accs,
        "contra_mean": np.mean(contras), "contra_std": np.std(contras, ddof=1),
        "contra_vals": contras,
    }

agg = {name: aggregate(multi_seed_results[name]) for name in CONFIG_NAMES}

DISPLAY_NAMES = {
    "baseline": "Baseline", "mitr_infonce": "MITR-InfoNCE", "mitr_cosine": "MITR-Cosine",
    "mitr_cka": "MITR-CKA", "mitr_club": "MITR-CLUB",
}
COLORS = {
    "baseline": "#78909C", "mitr_infonce": "#9C27B0", "mitr_cosine": "#FF9800",
    "mitr_cka": "#4CAF50", "mitr_club": "#2196F3",
}
N_SEEDS = len(SEEDS)

# ── Paired t-tests ────────────────────────────────────────────

print("=" * 60)
print("STATISTICAL ANALYSIS — Paired t-tests (2-tailed)")
print("=" * 60)

bl_accs    = agg["baseline"]["acc_vals"]
bl_contras = agg["baseline"]["contra_vals"]

for strategy in MI_STRATEGIES:
    key = f"mitr_{strategy}"
    s_accs    = agg[key]["acc_vals"]
    s_contras = agg[key]["contra_vals"]

    diffs_acc = [a - b for a, b in zip(bl_accs, s_accs)]
    diffs_con = [a - b for a, b in zip(bl_contras, s_contras)]

    if all(d == 0 for d in diffs_acc):
        t_acc, p_acc = 0.0, 1.0
        acc_note = "(identical values)"
    else:
        t_acc, p_acc = stats.ttest_rel(bl_accs, s_accs)
        acc_note = "*" if p_acc < 0.05 else "(n.s.)"

    if all(d == 0 for d in diffs_con):
        t_contra, p_contra = 0.0, 1.0
        con_note = "(identical values)"
    else:
        t_contra, p_contra = stats.ttest_rel(bl_contras, s_contras)
        con_note = "*" if p_contra < 0.05 else "(n.s.)"

    agg[key]["acc_tstat"]    = t_acc
    agg[key]["acc_pval"]     = p_acc
    agg[key]["contra_tstat"] = t_contra
    agg[key]["contra_pval"]  = p_contra

    name = DISPLAY_NAMES[key]
    print(f"\n{name} vs Baseline:")
    print(f"  Accuracy:         t={t_acc:.3f}, p={p_acc:.4f}  {acc_note}")
    print(f"  Contradiction:    t={t_contra:.3f}, p={p_contra:.4f}  {con_note}")

# ── Summary table ─────────────────────────────────────────────

print("\n" + "=" * 60)
print(f"RESULTS SUMMARY — Mean ± Std across {N_SEEDS} seeds")
print("=" * 60)
print(f"\n{'Config':<18} {'Accuracy':>18} {'Contradiction Rate':>22}")
print("-" * 62)
for name in CONFIG_NAMES:
    a = agg[name]
    print(f"{DISPLAY_NAMES[name]:<18} {a['acc_mean']:.4f} ± {a['acc_std']:.4f}    {a['contra_mean']:.4f} ± {a['contra_std']:.4f}")

# ── Plot: bar charts with error bars ──────────────────────────

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    try:
        plt.style.use("seaborn-whitegrid")
    except OSError:
        pass

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x = np.arange(len(CONFIG_NAMES))
width = 0.5
colors = [COLORS[n] for n in CONFIG_NAMES]

# Accuracy
ax = axes[0]
means = [agg[n]["acc_mean"] for n in CONFIG_NAMES]
stds  = [agg[n]["acc_std"]  for n in CONFIG_NAMES]
bars = ax.bar(x, means, width, yerr=stds, capsize=6, color=colors,
              edgecolor="black", linewidth=0.8, error_kw={"linewidth": 1.5})
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY_NAMES[n] for n in CONFIG_NAMES], fontsize=10)
lo, hi = min(means) - max(stds) - 0.01, max(means) + max(stds) + 0.01
ax.set_ylim(max(0, lo), min(1, hi))
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title(f"Validation Accuracy (mean ± std, {N_SEEDS} seeds)", fontsize=13, fontweight="bold")
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.002,
            f"{m:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

# Contradiction rate
ax = axes[1]
means_c = [agg[n]["contra_mean"] for n in CONFIG_NAMES]
stds_c  = [agg[n]["contra_std"]  for n in CONFIG_NAMES]
bars = ax.bar(x, means_c, width, yerr=stds_c, capsize=6, color=colors,
              edgecolor="black", linewidth=0.8, error_kw={"linewidth": 1.5})
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY_NAMES[n] for n in CONFIG_NAMES], fontsize=10)
lo2, hi2 = min(means_c) - max(stds_c) - 0.01, max(means_c) + max(stds_c) + 0.01
ax.set_ylim(max(0, lo2), min(1, hi2))
ax.set_ylabel("Contradiction Rate", fontsize=12)
ax.set_title(f"Contradiction Rate (mean ± std, {N_SEEDS} seeds)\n(lower = better)", fontsize=13, fontweight="bold")
for bar, m, s in zip(bars, means_c, stds_c):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.002,
            f"{m:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

fig.suptitle("Part D: Multi-Seed Statistical Validation — DistilBERT on BoolQ",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()

out_path = os.path.join(CFG.output_dir, "multi_seed_validation.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nFigure saved -> {out_path}")

# ── Plot: per-seed line graphs ────────────────────────────────

MARKERS = {
    "baseline": "o", "mitr_infonce": "D", "mitr_cosine": "^",
    "mitr_cka": "P", "mitr_club": "s",
}

fig2, axes2 = plt.subplots(1, 2, figsize=(14, 6))
seed_indices = list(range(1, N_SEEDS + 1))

# Accuracy per seed
ax = axes2[0]
for name in CONFIG_NAMES:
    vals = agg[name]["acc_vals"]
    ax.plot(seed_indices, vals, color=COLORS[name], marker=MARKERS[name],
            label=DISPLAY_NAMES[name], linewidth=2, markersize=8)
    ax.axhline(y=agg[name]["acc_mean"], color=COLORS[name], linestyle="--", alpha=0.4)
ax.set_xticks(seed_indices)
ax.set_xticklabels([str(s) for s in SEEDS], fontsize=10)
ax.set_xlabel("Seed", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title(f"Accuracy by Seed ({N_SEEDS} seeds)", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="best")

# Contradiction rate per seed
ax = axes2[1]
for name in CONFIG_NAMES:
    vals = agg[name]["contra_vals"]
    ax.plot(seed_indices, vals, color=COLORS[name], marker=MARKERS[name],
            label=DISPLAY_NAMES[name], linewidth=2, markersize=8)
    ax.axhline(y=agg[name]["contra_mean"], color=COLORS[name], linestyle="--", alpha=0.4)
ax.set_xticks(seed_indices)
ax.set_xticklabels([str(s) for s in SEEDS], fontsize=10)
ax.set_xlabel("Seed", fontsize=12)
ax.set_ylabel("Contradiction Rate", fontsize=12)
ax.set_title(f"Contradiction Rate by Seed ({N_SEEDS} seeds)\n(lower = better)", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="best")

fig2.suptitle("Per-Seed Results — DistilBERT on BoolQ",
              fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()

out_path2 = os.path.join(CFG.output_dir, "multi_seed_per_seed.png")
plt.savefig(out_path2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Per-seed figure saved -> {out_path2}")

# ── Save aggregated stats ─────────────────────────────────────
agg_json = os.path.join(CFG.output_dir, "multi_seed_aggregated.json")
agg_save = {}
for name in CONFIG_NAMES:
    a = agg[name]
    entry = {
        "acc_mean": a["acc_mean"], "acc_std": a["acc_std"], "acc_vals": a["acc_vals"],
        "contra_mean": a["contra_mean"], "contra_std": a["contra_std"], "contra_vals": a["contra_vals"],
    }
    if name != "baseline":
        entry["acc_pval"]    = a.get("acc_pval")
        entry["contra_pval"] = a.get("contra_pval")
    agg_save[name] = entry

with open(agg_json, "w") as fh:
    json.dump(agg_save, fh, indent=2, default=float)
print(f"Aggregated stats saved -> {agg_json}")

## Discussion (147 Words)

Across 5 seeds and all 4 MI strategies, no MITR configuration significantly improves DistilBERT on either accuracy or contradiction rate. InfoNCE (p=0.74 accuracy, p=0.51 contradiction), Cosine (p=0.31, p=0.48), and CKA (p=0.17, p=0.41) are all statistically indistinguishable from Baseline. CLUB is the sole significant result: it *hurts* accuracy (0.6875 vs 0.6963, p=0.027), confirming its MI loss divergence (hitting the -50 clamp at every seed) is a consistent failure mode, not a seed artifact. Notably, CKA — excluded from our initial scoped-down design as the "weakest" based on single-seed results — actually achieved the highest mean accuracy (0.6980), illustrating precisely why single-seed rankings are unreliable. These results directly confirm the Part A critique: the original DistilBERT claims were within normal run-to-run variance (std 0.003–0.007). The critical remaining test is whether the larger RoBERTa effect sizes (+1.93% accuracy, -3.20% contradiction rate) survive multi-seed validation.

### What the full experiment would look like

5 seeds × all 4 strategies × 3 backbones (DistilBERT, BERT, RoBERTa) × 5 epochs ≈ 15–20 GPU hours on T4. The DistilBERT portion is now complete. The critical remaining test is BERT and RoBERTa, where the effect sizes were larger and the backbone-dependent reversal (MITR helps RoBERTa, hurts BERT) is the paper's central finding.